# Overview

[Imagen en Vertex AI](https://cloud.google.com/vertex-ai/docs/generative-ai/image/overview) lleva las capacidades de IA generativa de vanguardia de Google a los desarrolladores de aplicaciones. Con Imagen en Vertex AI, los desarrolladores de aplicaciones pueden crear productos de IA de próxima generación que transforman la imaginación de sus usuarios en activos visuales de alta calidad, en segundos.

Con Imagen, puedes hacer lo siguiente:
- Generar imágenes nuevas a partir de un prompt.
- Editar una imagen completa cargada o generada con una indicación de texto.
- Editar solo partes de una imagen cargada o generada usando un área de máscara que definas.
- Aumentar la resolución de imágenes existentes, generadas o editadas.
- Ajustar un modelo con un tema específico (por ejemplo, un bolso o zapato específico) para la generación de imágenes.
- Obtener descripciones de texto de imágenes con subtítulos visuales.
- Obtener respuestas a una pregunta sobre una imagen con Visual Question Answering (VQA).

# Objetivos

En este cuaderno, exploraremos algunas de las funciones de **generación de imágenes** de Imagen utilizando el SDK de Python de Vertex AI.


Vamos a realizar diversos ejercicios
## Generación de imágenes
1. Generar imágenes utilizando prompts de texto
2. Experimentar con diferentes parámetros:
    - número de imágenes a generar
    - reproducir las mismas imágenes generadas a partir de un prompt utilizando una semilla
    - influenciar las imágenes generadas usanndo prompts negativos
3. Comparar la calidad de los diferentes modelos
4. Realizar llamadas programáticas utilizando modelos pre-entrenados vs el cliente de GenAI

### Costes

- Este notebook utiliza componentes de Google Cloud que generan costes:
  - Vertex AI (Imagen)

- Descubre sobre [el coste de Vertex AI](https://cloud.google.com/vertex-ai/pricing) y utiliza la [Calculadora de precios](https://cloud.google.com/products/calculator/) para generar una estimación de precios basado en el uso de recursos planificado.

### Instalar el SDK de Python de Vertex AI

In [ ]:
%pip install --upgrade --quiet google-genai

### Autentica tu entorno de cuaderno (solo Colab)

Si estás ejecutando este cuaderno en Google Colab, ejecuta la siguiente celda para autenticar tu entorno.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Importa las librerias necesarias

In [ ]:
from google import genai
from google.genai import types

### Inicializar Vertex AI en nuestro Google Cloud Project

In [ ]:
# Importar Vertex AI
import vertexai

# Definir la info del proyecto
PROJECT_ID = ""  # @param {type:"string"}
# Vamos a utilizar esta localización por defecto
LOCATION = "us-central1"

# Inicializar el módulo
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

Vamos a verificar que todo está en orden y qué modo estamos utilizando

In [ ]:
if not client.vertexai:
    print("Usando Gemini Developer API.")
elif client._api_client.project:
    print(
        f"Usando Vertex AI en el proyecto: {client._api_client.project} en la localización: {client._api_client.location}"
    )
elif client._api_client.api_key:
    print(
        f"Usando Vertex AI en modo express con API key: {client._api_client.api_key[:5]}...{client._api_client.api_key[-5:]}"
    )

###  Funciones auxiliares

---



La función `generate_images` acepta parametros adicionales que pueden influir en el número de imágenes generadas.

La función auxiliar `display_images_in_grid` es una herramienta auxiliar diseñada para mostrar una lista de imágenes en un formato de cuadrícula, organizando un máximo de cuatro imágenes por fila.

In [ ]:
import typing
import IPython.display
import math
import matplotlib.pyplot as plt

from PIL import Image as PIL_Image
from PIL import ImageOps as PIL_ImageOps

def display_image(
    image,
    max_width: int = 700,
    max_height: int = 400,
) -> None:
    pil_image = typing.cast(PIL_Image.Image, image._pil_image)
    if pil_image.mode != "RGB":
        # RGB is supported by all Jupyter environments (e.g. RGBA is not yet)
        pil_image = pil_image.convert("RGB")
    image_width, image_height = pil_image.size
    if max_width < image_width or max_height < image_height:
        # Resize to display a smaller notebook image
        pil_image = pil_image = PIL_ImageOps.contain(pil_image, (max_width, max_height))
    IPython.display.display(pil_image)


import matplotlib.pyplot as plt
import math
import io
from PIL import Image

def display_images_in_grid(images, cols=4):
    n = len(images)
    rows = math.ceil(n / cols)

    plt.figure(figsize=(15, 5 * rows))

    for i, img_obj in enumerate(images):
        plt.subplot(rows, cols, i + 1)

        # --- CORRECCIÓN AQUÍ ---
        # Convertimos los bytes de la respuesta en una imagen de PIL
        image_data = Image.open(io.BytesIO(img_obj.image.image_bytes))

        plt.imshow(image_data)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

### Cargar los modelos de generación de imágenes

  - Imagen4 - `imagen-4.0-generate-001`
  - Imagen4 Fast - `imagen-4.0-fast-generate-001`
  - Imagen4 Ultra - `imagen-4.0-ultra-generate-001`

In [ ]:
imagen4_model = "imagen-4.0-generate-001"
imagen4_model_fast = "imagen-4.0-fast-generate-001"
imagen4_model_ultra = "imagen-4.0-ultra-generate-001"

### Ejercicio 1 - Generar una imagen con Imagen 4

La función `generate_image` se usa para generar imágenes.
Lo único que necesitamos es un prompt.

https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/imagen-api?hl=es-419#python_1


In [ ]:
prompt = """
Una foto de dos personas dandose la mano enfrente del Big Ben al atardecer
"""

response = client.models.generate_images(
    # TODO: Rellenar con los parámetros necesarios para generar imágenes
)

display_image(response.generated_images[0].image)


### Ejercicio 2 - Generar 4 imagenes a partir del mismo prompt

Prueba a ejecutar el código varias veces. Notarás como el modelo genera imágenes nuevas cada vez, incluso aunque estemos utilizando el mismo prompt.

In [ ]:
response = client.models.generate_images(
    model=imagen4_model,
    prompt=prompt,
    # TODO: Rellenar con los parámetros necesarios de configuración para generar 4 imágenes
)

display_images_in_grid(response.generated_images)

#### Ejercicio 3 - Crear las mismas imágenes con el mismo prompt

Con el parámetro `seed`, puedes influir en el modelo para que cree la misma salida a partir de la misma entrada cada vez. Ten en cuenta que el orden de las imágenes generadas aún podría cambiar. Para que la llamada funcione al usar el parámetro `seed`, la marca de agua debe estar deshabilitada.

Como Imagen3 e Imagen4 utilizan mejora automática del prompt por defecto, vamos a utilizar Imagen2 para ver sus resultados

In [ ]:
prompt="Un gato sentado en el alfeizar de una ventana, de noche."

response = client.models.generate_images(
    model=imagen4_model_fast,
    prompt=prompt,
    config=types.GenerateImagesConfig(
        number_of_images=4,
        # TODO: Rellenar con los parámetros necesarios
    )
)
display_images_in_grid(response.generated_images)

# Volvemos a ejecutar para ver que las imagenes son las mismas
response = client.models.generate_images(
    model=imagen4_model_fast,
    prompt=prompt,
    config=types.GenerateImagesConfig(
        number_of_images=4,
        # TODO: Rellenar con los parámetros necesarios
    )
)
display_images_in_grid(response.generated_images)

Como se puede observar, el modelo produjo un conjunto idéntico de imágenes después de utilizar el parámetro seed, aunque el orden de las imágenes varía

#### Ejercicio 4 - Crear imágenes verticales u horizontales


El parámetro `aspect_ratio` controla la relación de aspecto de la imagen. El valor predeterminado es "1:1".

Experimenta con el siguiente fragmento de código para generar imágenes verticales (*portrait*) y horizontales (*landscape*)

[Comprueba los Aspect Ratio disponibles](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/models/imagen/4-0-generate?hl=es-419)

In [ ]:
prompt="Una foto de gran angular de la Torre Eiffel, en Paris, de noche."

# Imagen 1:1
response = client.models.generate_images(
    model=imagen4_model_fast,
    prompt=prompt,
    config=types.GenerateImagesConfig(
        number_of_images=1,
        # TODO: Rellenar con los parámetros necesarios
    )
)
print("Foto Cuadrada")
display_image(response.generated_images[0].image)

# Imagen 16:9
response = client.models.generate_images(
    model=imagen4_model_fast,
    prompt=prompt,
    config=types.GenerateImagesConfig(
        number_of_images=1,
        # TODO: Rellenar con los parámetros necesarios
    )
)

print("Foto Horizontal")
display_image(response.generated_images[0].image)

# Imagen 4:5
response = client.models.generate_images(
    model=imagen4_model_fast,
    prompt=prompt,
    config=types.GenerateImagesConfig(
        number_of_images=1,
        # TODO: Rellenar con los parámetros necesarios
    )
)

print("Foto Vertical")
display_image(response.generated_images[0].image)

### Ejercicio 5: Imagen vs Imagen 4 Fast

Con Imagen 4, también tienes la opción de utilizar Imagen 4 Fast. Estas dos opciones de modelo te permiten elegir entre optimizar la calidad o la latencia, dependiendo de tu caso de uso.

**Imagen 4:** Genera imágenes de alta calidad con iluminación natural y un realismo fotográfico mejorado.

**Imagen 4 Fast:** Ideal para crear imágenes más brillantes con un mayor contraste.

#### Parámetros adicionales
También puedes configurar el parámetro `image_size` a `1K` o `2K`.
Al generar imágenes de personas, puedes ajustar los parámetros `safety_filter_level` y `person_generation` según corresponda:

- `person_generation`
  - `DONT_ALLOW` (No permitir)
  - `ALLOW_ADULT` (Permitir adultos)
  - `ALLOW_ALL` (Permitir todo)
- `safety_filter_level`
  - `BLOCK_LOW_AND_ABOVE` (Bloquear nivel bajo y superiores)
  - `BLOCK_MEDIUM_AND_ABOVE` (Bloquear nivel medio y superiores)
  - `BLOCK_ONLY_HIGH` (Bloquear solo nivel alto)
  - `BLOCK_NONE` (Ninguno / No bloquear)

In [ ]:
import matplotlib.pyplot as plt

prompt = """
an image of the New York skyline at sunset
"""

# Imagen 4 image generation
image = client.models.generate_images(
    model="", # TODO: Rellenar con los parámetros necesarios
    prompt=prompt,
    config=types.GenerateImagesConfig(
        number_of_images=1,
        aspect_ratio="3:4",
        image_size="2K",
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="ALLOW_ADULT",
    ),
)

# Imagen 4 Fast image generation
fast_image = client.models.generate_images(
    model="", # TODO: Rellenar con los parámetros necesarios
    prompt=prompt,
    config=types.GenerateImagesConfig(
        number_of_images=1,
        aspect_ratio="3:4",
        image_size="2K",
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="ALLOW_ADULT",
    ),
)

# Display generated images
fig, axis = plt.subplots(1, 2, figsize=(12, 6))
axis[0].imshow(image.generated_images[0].image._pil_image)
axis[0].set_title("Imagen 4")
axis[1].imshow(fast_image.generated_images[0].image._pil_image)
axis[1].set_title("Imagen 4 Fast")
for ax in axis:
    ax.axis("off")
plt.show()

### Ejercicio 6: Imágenes de alta calidad con Imagen 4 Ultra

Junto con Imagen 4 e Imagen 4 Fast, tienes la opción de utilizar Imagen 4 Ultra. Este modelo ofrece imágenes de una calidad excepcionalmente alta a cambio de una mayor latencia.

In [ ]:
prompt = """
Escena nocturna fotorrealista: una mirada al interior de un clásico diner estadounidense de los años 60, brillantemente iluminado, desde la fría calle exterior. Toda la vista se filtra a través de un gran panel de vidrio veteado por el agua de lluvia. El letrero de neón que dice "DINER" en el exterior proyecta reflejos coloridos sobre el pavimento mojado y en la propia ventana. Atmósfera melancólica y nostálgica, con una profundidad de campo reducida que enfatiza la superficie del cristal.
"""

image = client.models.generate_images(
    model="", # TODO: Rellenar con los parámetros necesarios
    prompt=prompt,
    config=types.GenerateImagesConfig(
        number_of_images=1,
        aspect_ratio="1:1",
        image_size="2K",
        safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
        person_generation="ALLOW_ADULT",
    ),
)

display_image(image.generated_images[0].image, max_width=1000, max_height=1000,)

### Ejercicio 7: Comparación de los distintos modelos de Imagen 4

Ahora que conocemos todos los modelos, vamos a realizar una comparación de la generación de los modelos para un mismo prompt

In [ ]:
import matplotlib.pyplot as plt

prompt = """
Escena nocturna fotorrealista: una mirada al interior de un clásico diner estadounidense de los años 60, brillantemente iluminado, desde la fría calle exterior. Toda la vista se filtra a través de un gran panel de vidrio veteado por el agua de lluvia. El letrero de neón que dice "DINER" en el exterior proyecta reflejos coloridos sobre el pavimento mojado y en la propia ventana. Atmósfera melancólica y nostálgica, con una profundidad de campo reducida que enfatiza la superficie del cristal.
"""

config = types.GenerateImagesConfig(
    number_of_images=1,
    aspect_ratio="1:1",
    image_size="2K",
    safety_filter_level="BLOCK_MEDIUM_AND_ABOVE",
    person_generation="ALLOW_ADULT",
)

# Imagen 4 image generation
image = client.models.generate_images(
    model=imagen4_model,
    prompt=prompt,
    config=config
)

# Imagen 4 Fast image generation
fast_image = client.models.generate_images(
    model=imagen4_model_fast,
    prompt=prompt,
    config=config
)

# Imagen 4 Ultra image generation
ultra_image = client.models.generate_images(
    model=imagen4_model_ultra,
    prompt=prompt,
    config=config
)

# Display generated images
fig, axis = plt.subplots(3, 1, figsize=(50, 50))
axis[0].imshow(image.generated_images[0].image._pil_image)
axis[0].set_title("Imagen 4")
axis[1].imshow(fast_image.generated_images[0].image._pil_image)
axis[1].set_title("Imagen 4 Fast")
axis[2].imshow(ultra_image.generated_images[0].image._pil_image)
axis[2].set_title("Imagen 4 ultra")

for ax in axis:
    ax.axis("off")
plt.show()